Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\1pasos_lstm_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [9]:
print(datosX[0])

[[-2.05135573  0.15144479 -0.85166941  0.03349964 -1.66085756         nan]
 [-2.07481311 -0.43307937 -0.21109699  0.01650173 -1.66085756         nan]
 [-2.0911524  -0.52770864 -0.28379117 -0.05506842 -1.66085756         nan]
 [-2.12221321 -0.65164785 -0.41118591 -0.17405378 -1.66085756         nan]
 [-2.15456823 -0.77474965 -0.50763164 -0.2447293  -1.66085756         nan]
 [-2.16556893 -0.87272862 -0.59759968 -0.29303915 -1.66085756         nan]
 [-2.19986525 -0.95898362 -0.68109001 -0.32166721 -1.51634012         nan]
 [-2.22332263 -1.03518949 -0.74658674 -0.39681586 -1.51634012         nan]
 [-2.19387957 -1.12730648 -0.83223632 -0.46391287 -1.51634012         nan]
 [-2.22413151 -1.19597551 -0.88909611 -0.49969795 -1.51634012         nan]
 [-2.22008713 -1.24873342 -0.98842083 -0.52385287 -1.51634012         nan]
 [-2.22736701 -1.29730419 -1.03232523 -0.5614272  -1.51634012         nan]]


Se dividen los conjuntos de datos

In [10]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 12, 6)
Las dimensiones de testX son:  (10533, 12, 6)
Las dimensiones de valX son:  (5189, 12, 6)


In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [12]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [13]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1], testX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 11s - 189ms/step - ia: 0.2312 - loss: 0.9248 - mae: 0.7874 - rmse: 0.9603 - smape: 1.5159 - val_ia: 0.3242 - val_loss: 1.0010 - val_mae: 0.8461 - val_rmse: 0.9901 - val_smape: 1.9333

Epoch 2/128                                           

58/58 - 1s - 24ms/step - ia: 0.2224 - loss: 0.9042 - mae: 0.7793 - rmse: 0.9488 - smape: 1.5316 - val_ia: 0.3247 - val_loss: 1.0042 - val_mae: 0.8476 - val_rmse: 0.9916 - val_smape: 1.9267

Epoch 3/128                                           

58/58 - 1s - 20ms/step - ia: 0.2096 - loss: 0.8974 - mae: 0.7778 - rmse: 0.9469 - smape: 1.5487 - val_ia: 0.3241 - val_loss: 0.9977 - val_mae: 0.8446 - val_rmse: 0.9884 - val_smape: 1.9368

Epoch 4/128                                           

58/58 - 1s - 16ms/step - ia: 0.2053 - loss: 0.8859 - mae: 0.7737 - rmse: 0.9395 - smape: 1.5605 - val_ia: 0.3246 - val_loss: 0.9990 - val_mae: 0.8453 - val_rmse: 0.9891 - val_smape: 1.9322

Epoch 5/128   

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



461/461 - 27s - 59ms/step - ia: 0.8445 - loss: 0.0939 - mae: 0.2039 - rmse: 0.2680 - smape: 0.4640 - val_ia: 0.5496 - val_loss: 0.0431 - val_mae: 0.1513 - val_rmse: 0.1745 - val_smape: 0.4254

Epoch 2/128                                                                       

461/461 - 13s - 27ms/step - ia: 0.9291 - loss: 0.0198 - mae: 0.0988 - rmse: 0.1324 - smape: 0.2632 - val_ia: 0.6366 - val_loss: 0.0201 - val_mae: 0.0993 - val_rmse: 0.1194 - val_smape: 0.2622

Epoch 3/128                                                                       

461/461 - 20s - 43ms/step - ia: 0.9512 - loss: 0.0089 - mae: 0.0684 - rmse: 0.0898 - smape: 0.2023 - val_ia: 0.7014 - val_loss: 0.0111 - val_mae: 0.0733 - val_rmse: 0.0895 - val_smape: 0.1916

Epoch 4/128                                                                       

461/461 - 13s - 28ms/step - ia: 0.9616 - loss: 0.0055 - mae: 0.0536 - rmse: 0.0707 - smape: 0.1640 - val_ia: 0.7530 - val_loss: 0.0081 - val_mae: 0.0599 - val_rmse: 0.07

...

Epoch 1/128                                                                         

461/461 - 11s - 24ms/step - ia: 0.2973 - loss: 0.9043 - mae: 0.7740 - rmse: 0.9352 - smape: 1.4366 - val_ia: 0.2322 - val_loss: 0.5167 - val_mae: 0.6085 - val_rmse: 0.6357 - val_smape: 1.1650

Epoch 2/128                                                                         

461/461 - 6s - 13ms/step - ia: 0.6171 - loss: 0.3689 - mae: 0.4778 - rmse: 0.5929 - smape: 0.9220 - val_ia: 0.3016 - val_loss: 0.2491 - val_mae: 0.4321 - val_rmse: 0.4543 - val_smape: 0.8365

Epoch 3/128                                                                         

461/461 - 6s - 14ms/step - ia: 0.7179 - loss: 0.2398 - mae: 0.3886 - rmse: 0.4814 - smape: 0.7488 - val_ia: 0.3565 - val_loss: 0.1401 - val_mae: 0.3224 - val_rmse: 0.3422 - val_smape: 0.6796

Epoch 4/128                                                                         

461/461 - 6s - 14ms/step - ia: 0.7508 - loss: 0.1861 - mae: 0.3410 - rmse: 0.42

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

922/922 - 24s - 26ms/step - ia: 0.2264 - loss: 0.7980 - mae: 0.7351 - rmse: 0.8734 - smape: 1.7157 - val_ia: 0.1088 - val_loss: 0.9203 - val_mae: 0.8143 - val_rmse: 0.8264 - val_smape: 1.8753

Epoch 2/128                                                                            

922/922 - 10s - 11ms/step - ia: 0.2250 - loss: 0.7921 - mae: 0.7321 - rmse: 0.8692 - smape: 1.7063 - val_ia: 0.1092 - val_loss: 0.9084 - val_mae: 0.8094 - val_rmse: 0.8215 - val_smape: 1.8591

Epoch 3/128                                                                            

922/922 - 10s - 11ms/step - ia: 0.2329 - loss: 0.7817 - mae: 0.7272 - rmse: 0.8638 - smape: 1.6999 - val_ia: 0.1096 - val_loss: 0.8967 - val_mae: 0.8047 - val_rmse: 0.8167 - val_smape: 1.8414

Epoch 4/128                                                                            

922/922 - 9s - 10ms/step - ia: 0.2288 - loss: 0.7847 - mae: 0.727

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                            

58/58 - 8s - 137ms/step - ia: 0.3067 - loss: 0.7305 - mae: 0.6941 - rmse: 0.8480 - smape: 1.4113 - val_ia: 0.4453 - val_loss: 0.5973 - val_mae: 0.6622 - val_rmse: 0.7639 - val_smape: 1.3428

Epoch 2/128                                                                            

58/58 - 4s - 75ms/step - ia: 0.5682 - loss: 0.3716 - mae: 0.4858 - rmse: 0.6033 - smape: 1.0048 - val_ia: 0.6808 - val_loss: 0.2168 - val_mae: 0.4039 - val_rmse: 0.4578 - val_smape: 0.8400

Epoch 3/128                                                                            

58/58 - 3s - 44ms/step - ia: 0.7551 - loss: 0.1870 - mae: 0.3425 - rmse: 0.4310 - smape: 0.7105 - val_ia: 0.7522 - val_loss: 0.1742 - val_mae: 0.3529 - val_rmse: 0.4091 - val_smape: 0.7665

Epoch 4/128                                                                            

58/58 - 1s - 23ms/step - ia: 0.7863 - loss: 0.1567 - mae: 0.3114 - rmse: 

In [16]:
print(best)

{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}


In [17]:
#{'activation': 3, 'batch': 1, 'dropout': 0.0, 'layers': 4.0, 'learning_rate': 0.00027101707182655693, 'units': 4}